In [ ]:
# Load best model

import torch
from autoencoder.config_files_3L import EncoderConfigSettings_3Layers, DecoderConfigSettings_3Layers
from autoencoder.optimizer_settings import OptimizerSettings
from autoencoder.AE_3Layers import Encoder_3L, Decoder_3L, Autoencoder

study_name = "latent_space_2"
model_data_path = f"models/{study_name}_best_model.pt"

data = torch.load(model_data_path, weights_only=False)

best_hparams = data['hparams']

encoder_config = EncoderConfigSettings_3Layers(
  input_length=best_hparams["window_size"],
  input_channels=best_hparams["input_channels"],
  
  conv1_output_channels=best_hparams["conv1_channels"],
  conv1_kernel_size=best_hparams["conv1_kernel_size"],
  conv1_stride=best_hparams["conv1_stride"],
  activation_fn1=best_hparams["activation_fn1"],

  conv2_output_channels=best_hparams["conv2_channels"],
  conv2_kernel_size=best_hparams["conv2_kernel_size"],
  conv2_stride=best_hparams["conv2_stride"],
  activation_fn2=best_hparams["activation_fn2"],

  conv3_output_channels=best_hparams["conv3_channels"],
  conv3_kernel_size=best_hparams["conv3_kernel_size"],
  conv3_stride=best_hparams["conv3_stride"],
  activation_fn3=best_hparams["activation_fn3"],

  latent_size=best_hparams["latent_size"]
)

decoder_config = DecoderConfigSettings_3Layers(encoder_config)
optimizer_config = OptimizerSettings(lr=best_hparams["lr"], weight_decay=best_hparams["weight_decay"])

encoder = Encoder_3L(encoder_config)
decoder = Decoder_3L(decoder_config)

model = Autoencoder(encoder=encoder, decoder=decoder, optimizer_config=optimizer_config)

model.load_state_dict(data['state_dict'])
model.eval()

Autoencoder(
  (encoder): Encoder_3L(
    (conv1): Conv1d(1, 64, kernel_size=(11,), stride=(2,), padding=(5,))
    (activation1): LeakyReLU(negative_slope=0.01)
    (conv2): Conv1d(64, 64, kernel_size=(9,), stride=(1,), padding=(4,))
    (activation2): GELU(approximate='none')
    (conv3): Conv1d(64, 32, kernel_size=(9,), stride=(1,), padding=(4,))
    (activation3): LeakyReLU(negative_slope=0.01)
    (flatten): Flatten(start_dim=1, end_dim=-1)
    (latent): Linear(in_features=2240, out_features=2, bias=True)
  )
  (decoder): Decoder_3L(
    (latent_inv): Linear(in_features=2, out_features=2240, bias=True)
    (unflatten): Unflatten(dim=1, unflattened_size=(32, 70))
    (deconv3): ConvTranspose1d(32, 64, kernel_size=(9,), stride=(1,), padding=(4,))
    (activation3): LeakyReLU(negative_slope=0.01)
    (deconv2): ConvTranspose1d(64, 64, kernel_size=(9,), stride=(1,), padding=(4,))
    (activation2): GELU(approximate='none')
    (deconv1): ConvTranspose1d(64, 1, kernel_size=(11,), stride